In [ ]:
import unicodedata as uni
import pandas as pd
from src.extract import extrair

df = extrair("dados/Total Ovtrampas.xlsx", "CICLO 1")

In [2]:
def normalizar_nome(coluna:str) -> str:
    """'Agente / Matricula' -> 'Agente_Matricula'"""
    texto = uni.normalize('NFKD', str(coluna))
    texto = texto.encode("ascii", "ignore").decode("ascii")
    texto = texto.strip().lower()
    for caractere in "-/().$":
        texto = texto.replace(caractere, "_")
    while "__" in texto:
        texto = texto.replace("__","_")
    return texto.strip("_")


In [3]:
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]

In [4]:
df.columns = [normalizar_nome(c) for c in df.columns]

In [ ]:
for coluna in df.select_dtypes(include=["object", "str"]).columns:
    df[coluna] = df[coluna].str.strip()

df = df.rename(columns={"obs\nf _fechado_; e _extraviado_; r _recusado_; d _desocupado_; rec _recuperada": "obs"})

In [6]:
agen_matri = df["agente _ matricula"].str.split("/", n=1, expand=True)
df = df.drop(columns=["agente _ matricula"])

In [ ]:
df["ano"] = pd.to_numeric(df["ano"], errors="coerce")
df["ciclos"] = pd.to_numeric(df["ciclos"], errors="coerce")
df["ds"] = df["ds"].str.strip()
df["bairro"] = df["bairro"].str.strip()
df["bairro_area"] = df["bairro_area"].str.strip()
df["agente"] = agen_matri[0].str.strip()
df["matricula"] = agen_matri[1].str.strip()
df["qt"] = pd.to_numeric(df["qt"], errors="coerce")
df["id_ovt"] = df["id_ovt"].str.strip()
df["dt_coleta"] = pd.to_datetime(df["dt_coleta"], errors="coerce", format="ISO8601").dt.strftime("%Y-%m-%d")
df["dt_entrega_apoio"] = pd.to_datetime(df["dt_entrega_apoio"], errors="coerce", format="ISO8601").dt.strftime("%Y-%m-%d")
df["dt_entrega_lab"] = pd.to_datetime(df["dt_entrega_lab"], errors="coerce", format="ISO8601").dt.strftime("%Y-%m-%d")
df["dt_leitura"] = pd.to_datetime(df["dt_leitura"], errors="coerce", format="ISO8601").dt.strftime("%Y-%m-%d")
df["nm_tec_lab"] = df["nm_tec_lab"].str.strip()
df["dt_dig"] = pd.to_datetime(df["dt_dig"], errors="coerce", format="ISO8601").dt.strftime("%Y-%m-%d")
df["dt_envio_ds"] = pd.to_datetime(df["dt_envio_ds"], errors="coerce", format="ISO8601").dt.strftime("%Y-%m-%d")
df["n_ovos"] = pd.to_numeric(df["n_ovos"], errors="coerce")
df["obs"] = df["obs"].str.strip()

In [ ]:
df = df.dropna(how='all')
df = df.drop_duplicates()